# Usage Metrics Snapshot

Snapshot the hidden **Report Usage Metrics Model** semantic model into Lakehouse Delta tables for **unlimited history**.

## Why
Power BI's built-in usage metrics dataset only retains **30 days** of activity (rolling window). By querying it daily over XMLA via `sempy` and appending the rows to Lakehouse tables, we accumulate full history we control.

## Prerequisites
- Workspace on Premium / PPU / Fabric capacity (XMLA endpoint enabled).
- The `Report Usage Metrics Model` dataset exists in the workspace. It is auto-created the first time anyone clicks *More options → View usage metrics report* on a report.
- A Lakehouse attached to this notebook (any Lakehouse in the workspace).
- Tenant setting *“Usage metrics for content creators”* enabled.

## How to operate
1. Set `WORKSPACE_ID` below.
2. Run all cells once to verify.
3. Schedule the notebook **daily after 04:00 UTC** (the source refreshes around 03:00 UTC).

## What lands in the Lakehouse
One Delta table per source table, prefixed `usage_metrics_*`, with an extra `SnapshotUtc` column. Append-only.

| Source table         | Lakehouse table                       | Role          |
|----------------------|---------------------------------------|---------------|
| `Views`              | `usage_metrics_views`                 | Fact          |
| `Reports`            | `usage_metrics_reports`               | Dimension     |
| `Users`              | `usage_metrics_users`                 | Dimension     |
| `Dates`              | `usage_metrics_dates`                 | Dimension     |
| `DistributionMethods`| `usage_metrics_distributionmethods`   | Dimension     |
| `Platforms`          | `usage_metrics_platforms`             | Dimension     |

## 1. Configuration

In [ ]:
import sempy.fabric as fabric
from datetime import datetime, timezone, timedelta

# ---- EDIT ME --------------------------------------------------------------
WORKSPACE_ID = "<your-workspace-guid>"   # e.g., da2e15a8-c06d-4da0-ad10-c68aba63e564
DATASET_NAME = "Report Usage Metrics Model"

TABLES = [
    "Views",                # fact
    "Reports",              # dimensions
    "Users",
    "Dates",
    "DistributionMethods",
    "Platforms",
]

# How many days of snapshots to keep (set to None to keep forever)
RETENTION_DAYS = 400
# ---------------------------------------------------------------------------

snap_ts = datetime.now(timezone.utc)
print(f"Snapshot timestamp: {snap_ts.isoformat()}")

## 2. Snapshot all tables

Each table is pulled with `EVALUATE 'TableName'` (full table scan). Column names are cleaned up (DAX returns them as `'Table'[Column]`). The result is appended to the corresponding Lakehouse Delta table with `mergeSchema=true` so new columns added by Microsoft over time are tolerated.

In [ ]:
results = []

for t in TABLES:
    df = fabric.evaluate_dax(
        dataset=DATASET_NAME,
        workspace=WORKSPACE_ID,
        dax_string=f"EVALUATE '{t}'",
    )

    # Strip table name prefix from column names (DAX returns "TableName[Column]")
    df.columns = [c.split("[")[-1].rstrip("]") if "[" in c else c for c in df.columns]
    df["SnapshotUtc"] = snap_ts.isoformat()

    table_name = f"usage_metrics_{t.lower()}"
    sdf = spark.createDataFrame(df)
    (sdf.write
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name))

    results.append((t, table_name, len(df)))
    print(f"  {t:25s} -> {table_name:40s} {len(df):>8} rows")

print("\nDone.")

## 3. Retention (optional)

Trim snapshots older than `RETENTION_DAYS` so the tables don't grow forever. Default 400 days ≈ 13 months — enough for YoY comparisons.

In [ ]:
if RETENTION_DAYS:
    cutoff = (snap_ts - timedelta(days=RETENTION_DAYS)).isoformat()
    for _, table_name, _ in results:
        spark.sql(f"DELETE FROM {table_name} WHERE SnapshotUtc < '{cutoff}'")
        print(f"  Trimmed {table_name} (< {cutoff})")
else:
    print("Retention disabled — keeping all snapshots.")

---
# Verification & query examples

The cells below are **not part of the daily job** — use them to verify the snapshot worked and to demo how to query the historical data.

## 4. Snapshot health check

In [ ]:
for _, table_name, _ in results:
    df = spark.sql(f"""
        SELECT '{table_name}'                  AS table_name,
               COUNT(*)                        AS total_rows,
               COUNT(DISTINCT SnapshotUtc)     AS snapshot_count,
               MIN(SnapshotUtc)                AS first_snapshot,
               MAX(SnapshotUtc)                AS latest_snapshot
        FROM {table_name}
    """)
    df.show(truncate=False)

## 5. Peek at the latest snapshot of `Views`

In [ ]:
display(spark.sql("""
    WITH latest AS (
        SELECT MAX(SnapshotUtc) AS ts FROM usage_metrics_views
    )
    SELECT v.*
    FROM usage_metrics_views v
    JOIN latest ON v.SnapshotUtc = latest.ts
    LIMIT 20
"""))

## 6. Daily views per report (full history)

De-duplication pattern: take the **latest snapshot per natural key** so overlapping 30-day windows don't double-count. Adjust the `PARTITION BY` columns to whatever the real key columns are in your `Views` table (inspect with the previous cell).

In [ ]:
display(spark.sql("""
    WITH ranked AS (
        SELECT v.*,
               ROW_NUMBER() OVER (
                   PARTITION BY Date, ReportGuid, UserGuid
                   ORDER BY SnapshotUtc DESC
               ) AS rn
        FROM usage_metrics_views v
    )
    SELECT Date,
           ReportGuid,
           COUNT(*)                  AS views,
           COUNT(DISTINCT UserGuid)  AS distinct_users
    FROM ranked
    WHERE rn = 1
    GROUP BY Date, ReportGuid
    ORDER BY Date DESC, views DESC
"""))

## 7. Top reports last 30 days (joined to `Reports` dim)

In [ ]:
display(spark.sql("""
    WITH ranked_views AS (
        SELECT v.*,
               ROW_NUMBER() OVER (
                   PARTITION BY Date, ReportGuid, UserGuid
                   ORDER BY SnapshotUtc DESC
               ) AS rn
        FROM usage_metrics_views v
    ),
    latest_reports AS (
        SELECT r.*,
               ROW_NUMBER() OVER (
                   PARTITION BY ReportGuid
                   ORDER BY SnapshotUtc DESC
               ) AS rn
        FROM usage_metrics_reports r
    )
    SELECT r.DisplayName,
           COUNT(*)                  AS views,
           COUNT(DISTINCT v.UserGuid) AS distinct_users
    FROM ranked_views v
    LEFT JOIN latest_reports r
      ON r.ReportGuid = v.ReportGuid AND r.rn = 1
    WHERE v.rn = 1
      AND v.Date >= date_sub(current_date(), 30)
    GROUP BY r.DisplayName
    ORDER BY views DESC
"""))